# RetailPulse: Phase 2 – Feature Engineering & Exploratory Data Analysis

## 1. Objectives
In Phase 2, we build analytical and predictive features from the 2-year cleaned transactions dataset:
- **Customer Features**: Recency, Frequency, Monetary, Average Order Value (AOV), Average Purchase Interval, and Tenure.
- **Time-Series Features**: Daily and weekly sales aggregates, lag features (7, 14, 28 days), rolling statistics, UK holiday flags, cyclical trigonometric features, and trend indices.
- **Leakage-Free Churn Dataset**: Explicit 90-day evaluation cutoff avoiding target leakage.
- **Interactive Visualizations**: High-dimensional EDA with Plotly.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root_dir))

from src.utils.config import load_config
from src.features.rfm import calculate_rfm_features
from src.features.lag_features import build_daily_time_series, build_lag_features
from src.features.churn_features import build_churn_features

cfg = load_config(str(root_dir / 'config.yaml'))
print("Config loaded successfully.")

## 2. Load Processed Datasets

In [ ]:
df_clean = pd.read_parquet(cfg['data']['cleaned_parquet'])
df_rfm = pd.read_parquet(root_dir / 'data' / 'processed' / 'customer_features.parquet')
df_daily = pd.read_parquet(root_dir / 'data' / 'processed' / 'daily_sales_features.parquet')
df_churn = pd.read_parquet(root_dir / 'data' / 'processed' / 'churn_features.parquet')

print(f"Clean Transactions: {len(df_clean):,}")
print(f"Customer RFM Profiles: {len(df_rfm):,}")
print(f"Daily Time Series: {len(df_daily):,} days")
print(f"Churn Cohort Size: {len(df_churn):,} customers (Churn Rate: {df_churn['Churn'].mean()*100:.1f}%)")

## 3. Customer RFM & Behavioral Analysis

In [ ]:
df_rfm[['Recency', 'Frequency', 'Monetary', 'AvgOrderValue', 'TenureDays', 'AvgPurchaseInterval']].describe().round(2)

## 4. Time Series Trend & Seasonality

In [ ]:
fig = px.line(
    df_daily,
    x='Date',
    y=['total_sales', 'total_sales_rolling_mean_7d', 'total_sales_rolling_mean_28d'],
    title='RetailPulse Daily Net Sales with 7-Day & 28-Day Moving Averages',
    labels={'value': 'Net Sales (£)', 'Date': 'Date', 'variable': 'Metric'},
    template='plotly_white'
)
fig.show()

## 5. Day of Week & Monthly Seasonality Patterns

In [ ]:
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_avg = df_daily.groupby('day_name')['total_sales'].mean().reindex(dow_order).reset_index()
px.bar(dow_avg, x='day_name', y='total_sales', title='Average Sales by Day of Week', template='plotly_white').show()

## 6. Summary of Feature Engineering
The engineered feature datasets are persisted to `data/processed/` and ready for modeling:
- `customer_features.parquet` $\to$ Phase 3 Customer Segmentation (RFM + KMeans)
- `daily_sales_features.parquet` $\to$ Phase 4 Demand Forecasting (Prophet, LSTM, Hybrid)
- `churn_features.parquet` $\to$ Phase 5 Churn Prediction (XGBoost, Random Forest)